# ML on PACE ICE: CPU and GPU

Run this notebook first in a CPU Jupyter session, then in a one-GPU Jupyter session. Follow Procedure 405 on the HAAG HPC Training site for setup and cleanup. The example adapts the public Wisconsin breast cancer classification exercise from PACE's *Applications of Machine Learning* workshop. It is a compute demonstration, not a clinical tool or a GPU benchmark.


## 1. Choose the device

Leave `RUN_ON` as `"CPU"` for the first session. Change it to `"GPU"` only after starting a GPU Jupyter session and verifying `nvidia-smi` in a terminal. The notebook stops if a GPU was requested but TensorFlow cannot see one.


In [ ]:
RUN_ON = "CPU"  # Change to "GPU" for the second session.
assert RUN_ON in ("CPU", "GPU")

import os
import time
import numpy as np
import tensorflow as tf
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("Slurm job ID:", os.environ.get("SLURM_JOB_ID", "not found"))
print("TensorFlow version:", tf.__version__)
print("Visible CPUs:", tf.config.list_physical_devices("CPU"))
print("Visible GPUs:", tf.config.list_physical_devices("GPU"))
if RUN_ON == "GPU":
    assert tf.config.list_physical_devices("GPU"), "No GPU visible to TensorFlow. Check the GPU session and setup before continuing."
DEVICE = "/GPU:0" if RUN_ON == "GPU" else "/CPU:0"
print("Requested device:", DEVICE)


## 2. Load and prepare the data

We split the dataset before scaling. The scaler learns only from the training split, which prevents information from the test split leaking into training.


In [ ]:
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42, stratify=data.target
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype("float32")
X_test = scaler.transform(X_test).astype("float32")
print("Training rows:", len(X_train), "Test rows:", len(X_test))
print("Features per row:", X_train.shape[1])
print("Labels:", list(data.target_names))


## 3. Train the same model

This is a small feed-forward neural network. We request the selected device for its TensorFlow operations. The run time below covers model training only; it does not include installing packages or waiting for a Slurm allocation.


In [ ]:
tf.keras.utils.set_random_seed(42)
tf.config.set_soft_device_placement(True)  # Allow CPU-only input operations during GPU training.
with tf.device(DEVICE):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(X_train.shape[1],)),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    start = time.perf_counter()
    history = model.fit(X_train, y_train, epochs=12, batch_size=32, verbose=0)
    training_seconds = time.perf_counter() - start
print("Requested device:", DEVICE)
print(f"Training time: {training_seconds:.2f} seconds")
print(f"Final training accuracy: {history.history['accuracy'][-1]:.3f}")


## 4. Check the results

Accuracy is the fraction of test rows classified correctly. The confusion matrix gives the counts for each actual and predicted class. Your exact numbers may differ slightly across hardware and software versions.


In [ ]:
with tf.device(DEVICE):
    probabilities = model.predict(X_test, verbose=0).ravel()
predictions = (probabilities >= 0.5).astype(int)
print(f"Test accuracy: {accuracy_score(y_test, predictions):.3f}")
print("Confusion matrix (rows: actual 0/1; columns: predicted 0/1):")
print(confusion_matrix(y_test, predictions))
print("Class 0:", data.target_names[0], "| Class 1:", data.target_names[1])


## 5. Compare the two sessions

Write down the device, training time, and test accuracy from each run in Procedure 405. This dataset is small, so a GPU may be slower than a CPU. Once you have the results, stop the interactive Jupyter sessions in Open OnDemand.
